In [1]:
"""
Build the time series for Exit Intention vs. CVE crisis analysis.

Mirrors build_q1_timeseries.py exactly, just swapping the BAT-threshold
columns for exit-intention-severity columns:
  n_exit_any       posts that day with exit_intention_class in
                   {exit_explicit, exit_contemplating}  (broader)
  n_exit_explicit  posts that day with exit_intention_class == exit_explicit
                   (stricter -- explicit intent only)

No total-post-volume file needed: every row in exit_annotated_pass1.csv is
already a burnout-positive post, so exit_rate_within_burnout (computed here
too) is a valid rate on its own, same logic as exit_rate_within_burnout in
the original Q1 unified-dataset script.

Outputs:
  exit_daily_series.csv    one row per calendar day, 2018-01-01 to present
  exit_weekly_series.csv   same, resampled to weekly sums

Columns in both:
  n_exit_any        count, broader exit-intention threshold
  n_exit_explicit   count, stricter exit-intention threshold
  n_total_posts     total burnout posts that day (the local denominator)
  exit_rate_any            n_exit_any / n_total_posts
  exit_rate_explicit       n_exit_explicit / n_total_posts
  n_cve_run1        CVEs that day with cvss_base_score > 9.5
  n_cve_run2        CVEs that day with cvss_base_score >= 10
"""

import pandas as pd
import numpy as np
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
EXIT_PATH = "/Users/nadia/Desktop/redditRun_june/exit_annotated_pass1.csv"
NVD_PATH = "/Users/nadia/Desktop/redditRun_june/nvd_chunks/nvd_merged_full.csv"
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q2_analysis/"


def build_daily_exit(exit_path):
    # Only load the columns we actually need -- avoids the slow/heavy free-text
    # columns (selftext, text, cleaned_text, exit_evidence_span, exit_reasoning)
    cols = ["id", "created_date", "exit_intention_class"]
    df = pd.read_csv(exit_path, usecols=cols)
    df["date"] = pd.to_datetime(df["created_date"], errors="coerce").dt.date

    print("exit_intention_class distribution (sanity check):")
    print(df["exit_intention_class"].value_counts(dropna=False).to_string())

    df["is_exit_any"] = df["exit_intention_class"].isin(["exit_explicit", "exit_contemplating"]).astype(int)
    df["is_exit_explicit"] = (df["exit_intention_class"] == "exit_explicit").astype(int)

    daily = df.groupby("date").agg(
        n_exit_any=("is_exit_any", "sum"),
        n_exit_explicit=("is_exit_explicit", "sum"),
        n_total_posts=("id", "count"),
    ).reset_index()

    daily["exit_rate_any"] = daily["n_exit_any"] / daily["n_total_posts"]
    daily["exit_rate_explicit"] = daily["n_exit_explicit"] / daily["n_total_posts"]

    return daily


def build_daily_nvd(nvd_path):
    df = pd.read_csv(nvd_path, usecols=["cve_id", "published", "cvss_base_score"])
    df["date"] = pd.to_datetime(df["published"], errors="coerce").dt.date

    df["n_cve_run1"] = (df["cvss_base_score"] > 9.5).astype(int)
    df["n_cve_run2"] = (df["cvss_base_score"] >= 10).astype(int)

    daily = df.groupby("date").agg(
        n_cve_run1=("n_cve_run1", "sum"),
        n_cve_run2=("n_cve_run2", "sum"),
    ).reset_index()
    return daily


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    print("Building daily exit-intention series...")
    daily_exit = build_daily_exit(EXIT_PATH)

    print("\nBuilding daily NVD series...")
    daily_nvd = build_daily_nvd(NVD_PATH)

    # Combine on a continuous daily calendar, zero-filling quiet days for the
    # count columns. Note: exit_rate_* columns are NOT zero-filled here since
    # a day with zero burnout posts has an undefined rate (0/0) -- left as
    # NaN on those days, which the Granger script's dropna handling will need
    # to deal with (see note printed below).
    start = min(daily_exit["date"].min(), daily_nvd["date"].min())
    end = max(daily_exit["date"].max(), daily_nvd["date"].max())
    full_index = pd.DataFrame({"date": pd.date_range(start, end, freq="D").date})

    daily = full_index.merge(daily_exit, on="date", how="left") \
                       .merge(daily_nvd, on="date", how="left")

    count_cols = ["n_exit_any", "n_exit_explicit", "n_total_posts", "n_cve_run1", "n_cve_run2"]
    for c in count_cols:
        daily[c] = daily[c].fillna(0)

    n_zero_post_days = (daily["n_total_posts"] == 0).sum()
    print(f"\nFull daily series: {len(daily)} days, {start} to {end}")
    print(f"Days with zero burnout posts (rate undefined, will be NaN): {n_zero_post_days}")

    daily_path = os.path.join(OUT_DIR, "exit_daily_series.csv")
    daily.to_csv(daily_path, index=False)
    print(f"Saved -> {daily_path}")

    # Weekly resample -- counts sum, rates recompute from summed counts
    # (NOT averaged from daily rates, which would be wrong when daily post
    # counts vary -- summing counts first and dividing once is correct)
    weekly = daily.copy()
    weekly["date"] = pd.to_datetime(weekly["date"])
    weekly = weekly.set_index("date")[count_cols].resample("W").sum().reset_index()
    weekly["exit_rate_any"] = weekly["n_exit_any"] / weekly["n_total_posts"].replace(0, np.nan)
    weekly["exit_rate_explicit"] = weekly["n_exit_explicit"] / weekly["n_total_posts"].replace(0, np.nan)

    weekly_path = os.path.join(OUT_DIR, "exit_weekly_series.csv")
    weekly.to_csv(weekly_path, index=False)
    print(f"Saved -> {weekly_path}")

    print("\nDone. Point granger_exit_analysis.py at exit_daily_series.csv next.")


if __name__ == "__main__":
    main()

Building daily exit-intention series...
exit_intention_class distribution (sanity check):
exit_intention_class
no_exit               125679
exit_contemplating      9008
exit_explicit           1210

Building daily NVD series...

Full daily series: 3071 days, 2018-01-01 to 2026-05-29
Days with zero burnout posts (rate undefined, will be NaN): 34
Saved -> /Users/nadia/Desktop/redditRun_june/q2_analysis/exit_daily_series.csv
Saved -> /Users/nadia/Desktop/redditRun_june/q2_analysis/exit_weekly_series.csv

Done. Point granger_exit_analysis.py at exit_daily_series.csv next.
